# Assignment 2 – Python/NumPy Basics
**Topics:** Data visualization, projection onto line/plane, covariance, correlation  
**Datasets:** Date Fruit Dataset (assigned) + Wheat Seeds Dataset (UCI)

## Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

---
# PART 1 – Date Fruit Dataset (2D and 3D tasks)

## Step 1 – Load dataset, convert to NumPy array

In [ ]:
# Read the xlsx file – place 'Date_Fruit_Datasets.xlsx' in the same folder
df_full = pd.read_excel('Date_Fruit_Datasets.xlsx')

print("Shape:", df_full.shape)
print("Columns:", list(df_full.columns))
df_full.head(3)

In [ ]:
# The last column is the class label; all others are real-valued features
label_col    = df_full.columns[-1]
feature_cols = list(df_full.columns[:-1])

# Store unique class names; build string→int mapping
class_names = list(df_full[label_col].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
print("Classes:", class_names)

# Convert features to NumPy array and build integer label vector
X_all      = df_full[feature_cols].to_numpy(dtype=float)  # (n, d)
labels_idx = df_full[label_col].map(class_to_idx).to_numpy(dtype=int)  # (n,)

num_classes = len(class_names)
n_samples   = X_all.shape[0]
print(f"Samples: {n_samples}, Features: {X_all.shape[1]}, Classes: {num_classes}")

In [ ]:
# Define per-class marker shapes and colors used throughout all plots
SHAPES_2D = ['circle', 'square', 'diamond', 'triangle-up',
             'cross', 'x', 'star', 'pentagon', 'hexagram']
SHAPES_3D = ['circle', 'square', 'diamond', 'cross',
             'x', 'circle-open', 'square-open', 'diamond-open', 'circle-dot']
COLORS = px.colors.qualitative.Plotly

---
## 2D Tasks

### Task 2D-1 – Scatter plot (first two dimensions, different shape per class)

In [ ]:
dim1 = X_all[:, 0]   # first dimension
dim2 = X_all[:, 1]   # second dimension

fig_2d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_2d.add_trace(go.Scatter(
        x=dim1[mask], y=dim2[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=7, color=COLORS[i % len(COLORS)]
        )
    ))

fig_2d.update_layout(
    title='2D Scatter – First Two Dimensions',
    xaxis_title=feature_cols[0],
    yaxis_title=feature_cols[1],
    legend_title='Class'
)
fig_2d.show()

### Task 2D-2 – Add class means (same shape, larger size) to the 2D plot

In [ ]:
fig_2d_mean = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # --- original data points ---
    fig_2d_mean.add_trace(go.Scatter(
        x=dim1[mask], y=dim2[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=7, color=COLORS[i % len(COLORS)]
        )
    ))

    # --- class mean: same shape, larger, black border ---
    mean_d1 = dim1[mask].mean()
    mean_d2 = dim2[mask].mean()
    fig_2d_mean.add_trace(go.Scatter(
        x=[mean_d1], y=[mean_d2],
        mode='markers', name=f'{name} mean',
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=18, color=COLORS[i % len(COLORS)],
            line=dict(width=2, color='black')
        )
    ))

fig_2d_mean.update_layout(
    title='2D Scatter + Class Means',
    xaxis_title=feature_cols[0],
    yaxis_title=feature_cols[1],
    legend_title='Class / Mean'
)
fig_2d_mean.show()

### Task 2D-3 – Centered version of the first two dimensions

In [ ]:
# --- Compute the global mean of dim1 and dim2 using vector formula (no loop) ---
# mean = (1/n) * 1^T * X   where 1 is an all-ones column vector
ones_vec = np.ones(n_samples)                            # shape (n,)
mean_d1  = (ones_vec @ dim1) / n_samples                 # scalar
mean_d2  = (ones_vec @ dim2) / n_samples                 # scalar

# Centered dimensions: subtract the global mean from each column
dim1_c = dim1 - mean_d1    # centered dim1, shape (n,)
dim2_c = dim2 - mean_d2    # centered dim2, shape (n,)

print(f"Mean dim1: {mean_d1:.4f}  ->  centered mean: {dim1_c.mean():.2e}")
print(f"Mean dim2: {mean_d2:.4f}  ->  centered mean: {dim2_c.mean():.2e}")

In [ ]:
# Plot centered data
fig_centered_2d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_centered_2d.add_trace(go.Scatter(
        x=dim1_c[mask], y=dim2_c[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=7, color=COLORS[i % len(COLORS)]
        )
    ))

fig_centered_2d.update_layout(
    title='Centered Data (2D)',
    xaxis_title=f'{feature_cols[0]} (centered)',
    yaxis_title=f'{feature_cols[1]} (centered)',
    legend_title='Class'
)
fig_centered_2d.show()

### Task 2D-4 – Add line l = span{[-1.75, 1.75]} to the centered plot  
Direction vector: **v** = [-1.75, 1.75].  
Line equation: x₁ = -x₂  (since -1.75 / 1.75 = -1)

In [ ]:
# The direction vector of the line
v = np.array([-1.75, 1.75])          # shape (2,)

# Unit vector along the line (used later for projection)
v_hat = v / np.sqrt(v @ v)           # v / ||v||

# Build two end-points for drawing the line across the data range
# The line passes through the origin (centered data) with slope -1 (x1 = -x2)
t_range = np.array([-1, 1]) * max(np.abs(dim1_c).max(), np.abs(dim2_c).max())
line_x  = t_range              # x-coordinates of the two end-points
line_y  = -t_range             # y = -x  since x1 = -x2

fig_line_2d = go.Figure()

# Centered data points
for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_line_2d.add_trace(go.Scatter(
        x=dim1_c[mask], y=dim2_c[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=7, color=COLORS[i % len(COLORS)]
        )
    ))

# The line l
fig_line_2d.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines', name='l: x₁ = -x₂',
    line=dict(color='black', width=2, dash='dash')
))

# Annotate the line equation on the plot
fig_line_2d.add_annotation(
    x=line_x[1] * 0.7, y=line_y[1] * 0.7,
    text='l: x₁ = −x₂',
    showarrow=False, font=dict(size=13, color='black')
)

fig_line_2d.update_layout(
    title='Centered Data + Line l = span{[−1.75, 1.75]}',
    xaxis_title=f'{feature_cols[0]} (centered)',
    yaxis_title=f'{feature_cols[1]} (centered)',
    legend_title='Class'
)
fig_line_2d.show()

### Task 2D-5 – Project each data point onto line l; plot projected points (smaller shape) on the line

In [ ]:
# -----------------------------------------------------------------------
# Projection of a point x onto a line spanned by unit vector v_hat:
#   proj_x = (x · v_hat) * v_hat
# -----------------------------------------------------------------------

# Stack centered dim1 and dim2 into an (n, 2) matrix
X2_c = np.column_stack([dim1_c, dim2_c])   # shape (n, 2)

# Scalar projections: dot product of each row with v_hat -> shape (n,)
scalar_proj = X2_c @ v_hat    # (n,2) @ (2,) = (n,)

# Vector projections: each scalar * v_hat -> shape (n, 2)
proj_2d = scalar_proj[:, np.newaxis] * v_hat   # broadcasting: (n,1) * (2,)

proj_dim1 = proj_2d[:, 0]   # projected x-coordinates
proj_dim2 = proj_2d[:, 1]   # projected y-coordinates

In [ ]:
# Add projected points (smaller markers) to the existing centered + line plot
fig_proj_2d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # --- original centered points ---
    fig_proj_2d.add_trace(go.Scatter(
        x=dim1_c[mask], y=dim2_c[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=7, color=COLORS[i % len(COLORS)]
        )
    ))

    # --- projected points on the line (same shape, smaller size) ---
    fig_proj_2d.add_trace(go.Scatter(
        x=proj_dim1[mask], y=proj_dim2[mask],
        mode='markers', name=f'{name} proj',
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=4, color=COLORS[i % len(COLORS)],
            line=dict(width=1, color='black')
        )
    ))

# The line l
fig_proj_2d.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines', name='l: x₁ = −x₂',
    line=dict(color='black', width=2, dash='dash')
))
fig_proj_2d.add_annotation(
    x=line_x[1] * 0.7, y=line_y[1] * 0.7,
    text='l: x₁ = −x₂', showarrow=False,
    font=dict(size=13, color='black')
)

fig_proj_2d.update_layout(
    title='Centered Data + Line l + Projections (smaller shapes on line)',
    xaxis_title=f'{feature_cols[0]} (centered)',
    yaxis_title=f'{feature_cols[1]} (centered)',
    legend_title='Class'
)
fig_proj_2d.show()

---
## 3D Tasks

### Task 3D-1 – 3D Scatter plot (first three dimensions, different shape per class)

In [ ]:
dim1_3 = X_all[:, 0]
dim2_3 = X_all[:, 1]
dim3_3 = X_all[:, 2]

fig_3d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_3d.add_trace(go.Scatter3d(
        x=dim1_3[mask], y=dim2_3[mask], z=dim3_3[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=4, color=COLORS[i % len(COLORS)]
        )
    ))

fig_3d.update_layout(
    title='3D Scatter – First Three Dimensions',
    scene=dict(
        xaxis_title=feature_cols[0],
        yaxis_title=feature_cols[1],
        zaxis_title=feature_cols[2]
    ),
    legend_title='Class'
)
fig_3d.show()

### Task 3D-2 – Add 3D class means (same shape, larger) to the 3D plot

In [ ]:
fig_3d_mean = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    fig_3d_mean.add_trace(go.Scatter3d(
        x=dim1_3[mask], y=dim2_3[mask], z=dim3_3[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=4, color=COLORS[i % len(COLORS)]
        )
    ))

    # class mean in 3D
    m1 = dim1_3[mask].mean()
    m2 = dim2_3[mask].mean()
    m3 = dim3_3[mask].mean()

    fig_3d_mean.add_trace(go.Scatter3d(
        x=[m1], y=[m2], z=[m3],
        mode='markers', name=f'{name} mean',
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=12, color=COLORS[i % len(COLORS)],
            line=dict(width=2, color='black')
        )
    ))

fig_3d_mean.update_layout(
    title='3D Scatter + Class Means',
    scene=dict(
        xaxis_title=feature_cols[0],
        yaxis_title=feature_cols[1],
        zaxis_title=feature_cols[2]
    ),
    legend_title='Class / Mean'
)
fig_3d_mean.show()

### Task 3D-3 – Centered version of first three dimensions

In [ ]:
# Global mean of each dimension using vector formula
ones_n  = np.ones(n_samples)
mean_3d = (ones_n @ X_all[:, :3]) / n_samples   # shape (3,)
print("3D global mean:", mean_3d)

# Centered data: subtract mean from each row (broadcasting)
X3_c = X_all[:, :3] - mean_3d    # shape (n, 3)

dim1_c3 = X3_c[:, 0]
dim2_c3 = X3_c[:, 1]
dim3_c3 = X3_c[:, 2]

print("Centered means (should be ~0):", X3_c.mean(axis=0))

In [ ]:
# Plot centered 3D data
fig_3d_cent = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_3d_cent.add_trace(go.Scatter3d(
        x=dim1_c3[mask], y=dim2_c3[mask], z=dim3_c3[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=4, color=COLORS[i % len(COLORS)]
        )
    ))

fig_3d_cent.update_layout(
    title='Centered Data (3D)',
    scene=dict(
        xaxis_title=f'{feature_cols[0]} (centered)',
        yaxis_title=f'{feature_cols[1]} (centered)',
        zaxis_title=f'{feature_cols[2]} (centered)'
    ),
    legend_title='Class'
)
fig_3d_cent.show()

### Task 3D-4 – Add yellow plane spanned by **u₁** = [1, -2, 1]ᵀ and **u₂** = [2, 1, 0]ᵀ

In [ ]:
# The two spanning vectors of the plane
u1 = np.array([1.0, -2.0,  1.0])
u2 = np.array([2.0,  1.0,  0.0])

# Normal to the plane = u1 × u2  (cross product)
normal = np.cross(u1, u2)   # shape (3,)
print("Plane normal n = u1 × u2 =", normal)

# -----------------------------------------------------------------------
# Plane equation: n · (r - r0) = 0  where r0 = origin (centered data)
# normal = [n0, n1, n2]  =>  n0*x + n1*y + n2*z = 0
# We build a meshgrid over x,y and solve for z = -(n0*x + n1*y) / n2
# -----------------------------------------------------------------------
spread  = max(np.abs(X3_c).max() * 0.8, 1.0)   # extent of the plane patch
grid_pts = 30
xs = np.linspace(-spread, spread, grid_pts)
ys = np.linspace(-spread, spread, grid_pts)
XX, YY = np.meshgrid(xs, ys)

# Solve for Z on the plane
if abs(normal[2]) > 1e-10:
    ZZ = -(normal[0] * XX + normal[1] * YY) / normal[2]
else:
    # If n2 ≈ 0 use y instead
    ZZ = -(normal[0] * XX + normal[2] * np.zeros_like(XX)) / (normal[1] + 1e-10)

In [ ]:
fig_plane = go.Figure()

# Centered data points
for i, name in enumerate(class_names):
    mask = (labels_idx == i)
    fig_plane.add_trace(go.Scatter3d(
        x=dim1_c3[mask], y=dim2_c3[mask], z=dim3_c3[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=4, color=COLORS[i % len(COLORS)]
        )
    ))

# Yellow plane surface
fig_plane.add_trace(go.Surface(
    x=XX, y=YY, z=ZZ,
    colorscale=[[0, 'yellow'], [1, 'yellow']],  # solid yellow
    opacity=0.4,
    showscale=False,
    name='Plane span{u1, u2}'
))

fig_plane.update_layout(
    title='Centered 3D Data + Plane span{[1,-2,1], [2,1,0]}',
    scene=dict(
        xaxis_title=f'{feature_cols[0]} (centered)',
        yaxis_title=f'{feature_cols[1]} (centered)',
        zaxis_title=f'{feature_cols[2]} (centered)'
    ),
    legend_title='Class'
)
fig_plane.show()

### Task 3D-5 – Project each data point onto the plane; plot projections (smaller shapes) on the plane

In [ ]:
# -----------------------------------------------------------------------
# Projection onto a plane spanned by u1, u2 (through the origin):
#
# 1. Form the matrix A = [u1 | u2]  (3x2), columns are basis vectors
# 2. Projection matrix P = A (AᵀA)⁻¹ Aᵀ   (3x3)
# 3. Projected point x_proj = P @ x
#
# This is the standard least-squares orthogonal projection.
# -----------------------------------------------------------------------

A   = np.column_stack([u1, u2])          # shape (3, 2)
AtA = A.T @ A                            # shape (2, 2)

# Invert 2x2 AtA manually (no np.linalg.inv to keep it "basic")
# For a 2x2 [[a,b],[c,d]]: inv = 1/(ad-bc) * [[d,-b],[-c,a]]
a, b = AtA[0, 0], AtA[0, 1]
c, d = AtA[1, 0], AtA[1, 1]
det  = a * d - b * c
AtA_inv = (1.0 / det) * np.array([[d, -b], [-c, a]])   # shape (2, 2)

# Projection matrix P = A @ (AᵀA)⁻¹ @ Aᵀ   shape (3, 3)
P = A @ AtA_inv @ A.T

# Project all centered 3D points: (n,3) @ (3,3)ᵀ = (n,3)
# Note P is symmetric so P.T == P
proj_3d = X3_c @ P.T    # shape (n, 3)

print("First projected point:", proj_3d[0])

In [ ]:
# Add projected points (smaller shapes) to the plane plot
fig_proj_3d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # --- original centered points ---
    fig_proj_3d.add_trace(go.Scatter3d(
        x=dim1_c3[mask], y=dim2_c3[mask], z=dim3_c3[mask],
        mode='markers', name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=4, color=COLORS[i % len(COLORS)]
        )
    ))

    # --- projected points on the plane (same shape, smaller size) ---
    fig_proj_3d.add_trace(go.Scatter3d(
        x=proj_3d[mask, 0], y=proj_3d[mask, 1], z=proj_3d[mask, 2],
        mode='markers', name=f'{name} proj',
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=2, color=COLORS[i % len(COLORS)],
            line=dict(width=1, color='black')
        )
    ))

# Yellow plane
fig_proj_3d.add_trace(go.Surface(
    x=XX, y=YY, z=ZZ,
    colorscale=[[0, 'yellow'], [1, 'yellow']],
    opacity=0.4, showscale=False,
    name='Plane span{u1, u2}'
))

fig_proj_3d.update_layout(
    title='Centered 3D Data + Plane + Projections (smaller shapes on plane)',
    scene=dict(
        xaxis_title=f'{feature_cols[0]} (centered)',
        yaxis_title=f'{feature_cols[1]} (centered)',
        zaxis_title=f'{feature_cols[2]} (centered)'
    ),
    legend_title='Class'
)
fig_proj_3d.show()

---
# PART 2 – Wheat Seeds Dataset (Numeric Data Analysis)

## Step 1 – Load the seeds dataset

In [ ]:
# The seeds dataset is whitespace-separated with no header.
# 7 numeric attributes + 1 class label (integer 1/2/3) in the last column.
# Source: http://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt

seeds_url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt'

col_names = ['area', 'perimeter', 'compactness',
             'kernel_length', 'kernel_width',
             'asymmetry_coeff', 'groove_length', 'class']

df_seeds = pd.read_csv(seeds_url, sep='\s+', header=None, names=col_names)

print("Seeds dataset shape:", df_seeds.shape)
df_seeds.head(3)

In [ ]:
# Discard the class label column – we only need the 7 numeric attributes
S = df_seeds.drop(columns=['class']).to_numpy(dtype=float)   # shape (210, 7)
print("Feature matrix shape:", S.shape)

## Step 2 – Multivariate mean vector (vector formula)

In [ ]:
# Mean vector: mu = (1/N) * X^T * 1
# Where 1 is an all-ones vector of length N (number of rows)
N        = S.shape[0]                       # number of samples (210)
ones_N   = np.ones(N)                       # shape (N,)
mu       = (S.T @ ones_N) / N              # (7,N) @ (N,) -> (7,)  -- mean per attribute

print("Multivariate mean vector (one value per attribute):")
for attr, m in zip(col_names[:-1], mu):
    print(f"  {attr:20s}: {m:.4f}")

## Step 3 – Sample Covariance Matrix (via inner products of centered columns)

In [ ]:
# --- Center the data matrix ---
# Subtract the mean from each row: S_c = S - 1 * mu^T
# Broadcasting: (N,7) - (7,)  subtracts mu from every row
S_c = S - mu    # centered data matrix, shape (N, 7)

# --- Sample covariance matrix (Eq. 2.38) ---
# C = (1 / (N-1)) * S_c^T * S_c
# S_c^T @ S_c is the (7x7) matrix of inner products between centered columns
C = (S_c.T @ S_c) / (N - 1)   # shape (7, 7)

print("Sample Covariance Matrix (7x7):")
# Print as a formatted table
attr_labels = [c[:8] for c in col_names[:-1]]   # shortened labels for display
header = '         ' + '  '.join(f'{a:>10}' for a in attr_labels)
print(header)
for r, row in enumerate(C):
    row_str = f'{attr_labels[r]:>9}' + '  '.join(f'{v:10.4f}' for v in row)
    print(row_str)

## Step 4 – Correlation between Attributes 1 and 2  
Computed as the cosine of the angle between the two centered attribute vectors.

In [ ]:
# Centered attribute vectors (columns of S_c)
a1 = S_c[:, 0]   # centered attribute 1 (area),      shape (N,)
a2 = S_c[:, 1]   # centered attribute 2 (perimeter),  shape (N,)

# Cosine of the angle between a1 and a2:
#   cos(theta) = (a1 · a2) / (||a1|| * ||a2||)
#
# This equals the Pearson correlation coefficient between the two attributes.

dot_product = a1 @ a2                           # inner product (scalar)
norm_a1     = (a1 @ a1) ** 0.5                  # ||a1||  -- sqrt of dot with itself
norm_a2     = (a2 @ a2) ** 0.5                  # ||a2||

correlation = dot_product / (norm_a1 * norm_a2) # cosine similarity = correlation
angle_deg   = np.arccos(correlation) * 180 / np.pi

print(f"a1 · a2          = {dot_product:.4f}")
print(f"||a1||           = {norm_a1:.4f}")
print(f"||a2||           = {norm_a2:.4f}")
print(f"Correlation r₁₂  = cos(θ) = {correlation:.4f}")
print(f"Angle θ          = {angle_deg:.2f}°")

In [ ]:
# Scatter plot of Attribute 1 vs Attribute 2 (original values)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(S[:, 0], S[:, 1], s=20, alpha=0.6, color='steelblue', edgecolors='none')
ax.set_xlabel(col_names[0])
ax.set_ylabel(col_names[1])
ax.set_title(f'Scatter: {col_names[0]} vs {col_names[1]}\nCorrelation (cosine) = {correlation:.4f}')
plt.tight_layout()
plt.show()